In [ ]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────
# 1) LOAD & PRE-PROCESS
# ─────────────────────────────
df = pd.read_csv("Project 555.csv")
df["unit_users"] = df["unit_users"].str.replace(",", "").astype(float)

month_map = {
    "Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
    "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12
}
df["month_num"] = df["month"].map(month_map)
df["year_ad"]   = df["year"] - 543
df["date"]      = pd.to_datetime(dict(year=df["year_ad"],
                                      month=df["month_num"],
                                      day=1))
ts = (df
      .sort_values("date")
      .set_index("date")["unit_users"]
      .asfreq("MS"))                 # รายเดือนแน่นอน

# ─────────────────────────────
# 2) OPTIONAL – Quick accuracy check
#    (80 % / 20 %) ไม่แสดงค่าทำนาย
# ─────────────────────────────
train_size = int(len(ts)*0.8)
train, test = ts[:train_size], ts[train_size:]

tmp_model = SARIMAX(train,
                    order=(1,2,1),
                    seasonal_order=(1,1,1,12),
                    enforce_stationarity=False,
                    enforce_invertibility=False).fit(disp=False)

fc_test = tmp_model.forecast(steps=len(test))
mae  = mean_absolute_error(test, fc_test)
rmse = np.sqrt(mean_squared_error(test, fc_test))
mape = np.mean(np.abs((test - fc_test) / test))*100
print(f"TEST  MAE={mae:,.0f}  RMSE={rmse:,.0f}  MAPE={mape:.2f}%")

# ─────────────────────────────
# 3) REFIT บนข้อมูลทั้งหมด & FORECAST 12 เดือน
# ─────────────────────────────
full_model = SARIMAX(ts,
                     order=(1,2,1),
                     seasonal_order=(1,1,1,12),
                     enforce_stationarity=False,
                     enforce_invertibility=False).fit(disp=False)

future_steps = 12
fc_future    = full_model.forecast(steps=future_steps)
future_idx   = pd.date_range(start=ts.index[-1] + pd.offsets.MonthBegin(1),
                             periods=future_steps, freq="MS")
fc_future.index = future_idx

print("\n=== 12-Month Forecast ===")
print(fc_future)

# ─────────────────────────────
# 4) PLOT: Actual (historical) + Future forecast
# ─────────────────────────────
plt.figure(figsize=(12,5))
plt.plot(ts.index, ts, label="Actual", marker="o")
plt.plot(fc_future.index, fc_future, label="Forecast (Next 12 mo.)", marker="x")
plt.xlabel("Date"); plt.ylabel("Unit Users")
plt.title("SARIMA Forecast 12 Months Ahead")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()